In [ ]:
# English to French translation using two LSTMs: Encoder LSTM + Decoder LSTM

import tensorflow as tf
import numpy as np
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
****************************
****************************

# 1. Small English-French dataset for teaching
sentence_pairs = [
    ("i am happy", "je suis heureux"),
    ("i am sad", "je suis triste"),
    ("i am tired", "je suis fatigue"),
    ("i am hungry", "j ai faim"),
    ("i am thirsty", "j ai soif"),
    ("i love you", "je t aime"),
    ("i hate you", "je te deteste"),
    ("thank you", "merci"),
    ("good morning", "bonjour"),
    ("good night", "bonne nuit"),
    ("how are you", "comment allez vous"),
    ("where are you", "ou etes vous"),
    ("what is your name", "comment tu t appelles"),
    ("my name is john", "je m appelle john"),
    ("i live here", "je vis ici"),
    ("come here", "viens ici"),
    ("go away", "va t en"),
    ("sit down", "assieds toi"),
    ("stand up", "leve toi"),
    ("open the door", "ouvre la porte"),
    ("close the door", "ferme la porte"),
    ("i need help", "j ai besoin d aide"),
    ("help me", "aide moi"),
    ("i understand", "je comprends"),
    ("i do not understand", "je ne comprends pas"),
    ("i know", "je sais"),
    ("i do not know", "je ne sais pas"),
    ("this is good", "c est bon"),
    ("this is bad", "c est mauvais"),
    ("the weather is good", "il fait beau"),
    ("the weather is bad", "il fait mauvais"),
    ("i like this movie", "j aime ce film"),
    ("this movie is good", "ce film est bon"),
    ("this movie is bad", "ce film est mauvais"),
    ("i want water", "je veux de l eau"),
    ("i want food", "je veux de la nourriture"),
    ("i want coffee", "je veux du cafe"),
    ("i want tea", "je veux du the"),
    ("he is happy", "il est heureux"),
    ("she is happy", "elle est heureuse"),
    ("he is sad", "il est triste"),
    ("she is sad", "elle est triste"),
    ("we are ready", "nous sommes prets"),
    ("they are ready", "ils sont prets"),
    ("i am ready", "je suis pret"),
    ("are you ready", "etes vous pret"),
    ("see you later", "a plus tard"),
    ("see you tomorrow", "a demain"),
    ("i am learning", "j apprends"),
    ("i am learning french", "j apprends le francais")
]

# 2. Split English and French sentences
english_sentences = [pair[0] for pair in sentence_pairs]
french_sentences = [pair[1] for pair in sentence_pairs]

# 3. Clean text
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

english_sentences = [clean_text(sentence) for sentence in english_sentences]
french_sentences = [clean_text(sentence) for sentence in french_sentences]

# 4. Add start and end tokens to French target sentences
****************************

# 5. Tokenize English sentences
english_tokenizer = Tokenizer(oov_token="<OOV>")
english_tokenizer.fit_on_texts(english_sentences)

encoder_input_seq = english_tokenizer.texts_to_sequences(english_sentences)

english_vocab_size = len(english_tokenizer.word_index) + 1
max_encoder_len = max(len(seq) for seq in encoder_input_seq)

encoder_input_data = pad_sequences(encoder_input_seq, maxlen=max_encoder_len, padding="post")

# 6. Tokenize French sentences
french_tokenizer = Tokenizer(oov_token="<OOV>")
french_tokenizer.fit_on_texts(french_sentences)

decoder_full_seq = french_tokenizer.texts_to_sequences(french_sentences)

french_vocab_size = len(french_tokenizer.word_index) + 1
max_decoder_len = max(len(seq) for seq in decoder_full_seq)

****************************

# 7. Create decoder input and decoder target
# Example:
# Full French:     startseq je suis heureux endseq
# Decoder input:  startseq je suis heureux
# Decoder target: je suis heureux endseq
decoder_input_data = decoder_full_data[:, :-1]
decoder_target_data = decoder_full_data[:, 1:]

print("Encoder input shape:", encoder_input_data.shape)
print("Decoder input shape:", decoder_input_data.shape)
print("Decoder target shape:", decoder_target_data.shape)
print("English vocab size:", english_vocab_size)
print("French vocab size:", french_vocab_size)
print("Max encoder length:", max_encoder_len)
print("Max decoder length:", max_decoder_len)

# 8. Model parameters
embedding_dim = 64
lstm_units = 128

# 9. Encoder LSTM
encoder_inputs = Input(shape=(max_encoder_len,))

encoder_embedding = Embedding(input_dim=english_vocab_size, output_dim=embedding_dim, mask_zero=True)(encoder_inputs)

encoder_lstm = LSTM(lstm_units, return_state=True)

****************************

encoder_states = [state_h, state_c]

# 10. Decoder LSTM
decoder_inputs = Input(shape=(max_decoder_len - 1,))

decoder_embedding_layer = Embedding(input_dim=french_vocab_size, output_dim=embedding_dim, mask_zero=True)

decoder_embedding = decoder_embedding_layer(decoder_inputs)

decoder_lstm = LSTM(lstm_units, return_sequences=True, return_state=True)

****************************

****************************

decoder_outputs = decoder_dense(decoder_outputs)

# 11. Training model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

model.summary()

# 12. Train model
history = model.fit([encoder_input_data, decoder_input_data], decoder_target_data, epochs=300, batch_size=8, verbose=1)

# 13. Build inference encoder model
****************************

# 14. Build inference decoder model

# Create two empty input placeholders with the same shape as the LSTM states
decoder_state_input_h = Input(shape=(lstm_units,))
decoder_state_input_c = Input(shape=(lstm_units,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

# the decoder will receive one French token ID at a time
decoder_single_input = Input(shape=(1,))

# The decoder cannot understand raw token IDs directly.
# So this line turns a token ID into an embedding vector.
decoder_single_embedding = decoder_embedding_layer(decoder_single_input)
# decoder_embedding_layer is reused from training.

# This runs the decoder LSTM for one step.
****************************

# These states will be returned by the decoder model.
# Then they will be reused in the next prediction step.
decoder_states_outputs = [state_h2, state_c2]

#The LSTM output is not yet a word probability.
# The dense layer converts it into probabilities over every French word in the vocabulary.
decoder_single_outputs = decoder_dense(decoder_single_outputs)

# This creates the final inference decoder model.
# It takes: 1.current token 2.previous hidden state h 3.previous cell state c
# It returns: 1.predicted next-token probabilities 2.new hidden state h 3.new cell state c
# decoder_model(current_word, old_h, old_c) -> next_word_probabilities, new_h, new_c
decoder_model = Model([decoder_single_input] + decoder_states_inputs, [decoder_single_outputs] + decoder_states_outputs)
# [decoder_single_input] + decoder_states_inputs means [decoder_single_input] + [decoder_state_input_h, decoder_state_input_c]
# decoder_model = Model(inputs=[decoder_single_input, decoder_state_input_h, decoder_state_input_c],
    # outputs=[decoder_single_outputs, state_h2, state_c2])

# 15. Reverse dictionary for French token IDs
# convert predicted token IDs back into French words
index_to_french = {i: word for word, i in french_tokenizer.word_index.items()}

# get the numeric IDs for the special start and end tokens.
start_token_id = french_tokenizer.word_index["startseq"]
end_token_id = french_tokenizer.word_index["endseq"]

# 16. Translate one sentence
def translate_sentence(sentence): # This function takes one English sentence and returns a French translation.
    sentence = clean_text(sentence)

    input_seq = english_tokenizer.texts_to_sequences([sentence]) # convert the English sentence into token IDs.
    input_seq = pad_sequences(input_seq, maxlen=max_encoder_len, padding="post")

    # sends the English sentence through the encoder
    states_value = encoder_model.predict(input_seq, verbose=0)

    # The decoder starts with only one input: startseq
    target_seq = np.array([[start_token_id]])

    translated_words = [] # This list stores the predicted French words.

    for i in range(max_decoder_len): # This loop generates one French word at a time.
    # It stops when either: endseq is predicted or max_decoder_len is reached

        # Given the current token and current memory, what word comes next?
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)

        # The decoder outputs probabilities for all French tokens.
****************************

        if predicted_token_id == end_token_id:
            break

        # This converts the predicted token ID into an actual French word.
        predicted_word = index_to_french.get(predicted_token_id, "")
        # Find the French word for predicted_token_id.
        # If the ID does not exist, return an empty string instead.

        # This avoids adding unwanted tokens to the final translation.
        if predicted_word not in ["", "startseq", "endseq", "<OOV>"]:
            translated_words.append(predicted_word)

        # The predicted word becomes the next input.
        target_seq = np.array([[predicted_token_id]])

        # The decoder memory gets updated. This allows the decoder to remember what it has already generated.
****************************

    return " ".join(translated_words)

# 17. Try translation
test_sentences = [
    "i am happy",
    "i love you",
    "thank you",
    "where are you",
    "i want water",
    "this movie is good",
    "i love you"
]

for sentence in test_sentences:
    translation = translate_sentence(sentence)
    print("English:", sentence)
    print("French:", translation)
    print()

In [ ]:
encoder_input_seq

In [ ]:
# 17. Try translation
test_sentences = [
    "i am happy",
    "i love you",
    "thank you",
    "where are you",
    "i want water",
    "this movie is good"]

for sentence in test_sentences:
    translation = translate_sentence(sentence)
    print("English:", sentence)
    print("French:", translation)
    print()